# 🔍 Zero-Shot Defect Detection — WinCLIP Model Benchmarking

This notebook evaluates the **WinCLIP (Window-based CLIP)** zero-shot anomaly detection model on the **MVTec AD** dataset using GPU acceleration.

## What this notebook does
1. Downloads the **MVTec AD** dataset directly from Kaggle (~5 GB)
2. Loads the `openai/clip-vit-base-patch32` model on GPU
3. Runs sliding-window inference across all **15 MVTec AD categories**
4. Computes **AU-ROC** and **F1-Score** per category
5. Generates a comparative analysis JSON against state-of-the-art models
6. Saves `model_comparison.json` for download

---

## ❗ Before running this notebook

### Step 1: Set Colab runtime to GPU
`Runtime > Change runtime type > T4 GPU`

### Step 2: Get your Kaggle API Token
1. Go to [kaggle.com/settings](https://www.kaggle.com/settings)
2. Scroll to **API** section
3. Click **"Create New Token"** — copy the token string (starts with `KGAT_...`)
4. Paste it in **Cell 2** below where indicated

> **That's it!** You only need this notebook + your Kaggle API token string. Nothing else to upload. The 5 GB dataset downloads automatically.

---
## Cell 1: Install Required Packages

In [1]:
# ============================================================
# CELL 1: Install all required packages for Colab
# ============================================================
# Colab has torch, torchvision, numpy, Pillow, scikit-learn,
# and tqdm pre-installed. We install the remaining ones.
# ============================================================

!pip install -q transformers huggingface-hub ftfy regex scipy kaggle

print("\n\u2705 All packages installed successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.0 MB/s eta 0:00:00

✅ All packages installed successfully!


---
## Cell 2: Setup Kaggle API & Download MVTec AD Dataset

Paste your Kaggle API token below. The dataset will be downloaded and extracted automatically.

**Dataset source:** [kaggle.com/datasets/ipythonx/mvtec-ad](https://www.kaggle.com/datasets/ipythonx/mvtec-ad/data)

In [2]:
# ============================================================
# CELL 2: Setup Kaggle API & Download MVTec AD Dataset
# ============================================================

import os

# =====================================================
# ✏️ PASTE YOUR KAGGLE API TOKEN BELOW (between quotes)
# =====================================================
KAGGLE_TOKEN = "KGAT_57c712f166034d6ac0e6e852da7c226f"
# =====================================================

# --- Step 1: Set the Kaggle API token as environment variable ---
os.environ['KAGGLE_API_TOKEN'] = KAGGLE_TOKEN
print("\u2705 Kaggle API token configured!")

# --- Step 2: Download the MVTec AD dataset ---
DATASET_SLUG = 'ipythonx/mvtec-ad'
DOWNLOAD_DIR = '/content/mvtec_ad'

print(f"\n\u23f3 Downloading dataset: {DATASET_SLUG}")
print("   This is ~5 GB and may take 2-5 minutes...\n")

!kaggle datasets download -d {DATASET_SLUG} -p {DOWNLOAD_DIR} --force

print("\n\u2705 Download complete!")

# --- Step 3: Unzip the dataset ---
import glob

zip_files = glob.glob(os.path.join(DOWNLOAD_DIR, '*.zip'))
print(f"\n\u23f3 Extracting {len(zip_files)} zip file(s)...")

for zf in zip_files:
    print(f"   Extracting: {os.path.basename(zf)}")
    !unzip -q -o "{zf}" -d {DOWNLOAD_DIR}

print("\u2705 Extraction complete!")

# --- Step 4: Auto-detect the dataset root ---
# The dataset might extract as: mvtec_ad/bottle/... OR mvtec_ad/archive/bottle/...
# We auto-detect the correct path by looking for the category folders.

EXPECTED_CATEGORIES = {
    'bottle', 'cable', 'capsule', 'carpet', 'grid', 'hazelnut',
    'leather', 'metal_nut', 'pill', 'screw', 'tile',
    'toothbrush', 'transistor', 'wood', 'zipper'
}

def find_dataset_root(base_dir, expected_cats, max_depth=3):
    """Recursively search for the directory containing the category folders."""
    for root, dirs, _ in os.walk(base_dir):
        depth = root.replace(base_dir, '').count(os.sep)
        if depth > max_depth:
            break
        current_dirs = set(dirs)
        # Check if this directory contains at least 10 of the 15 expected categories
        overlap = current_dirs & expected_cats
        if len(overlap) >= 10:
            return root
    return None

detected_root = find_dataset_root(DOWNLOAD_DIR, EXPECTED_CATEGORIES)

if detected_root:
    found_cats = sorted([
        d for d in os.listdir(detected_root)
        if os.path.isdir(os.path.join(detected_root, d)) and d in EXPECTED_CATEGORIES
    ])
    print(f"\n\u2705 Dataset root auto-detected: {detected_root}")
    print(f"\u2705 Found {len(found_cats)} categories: {found_cats}")
else:
    print("\n\u274c Could not auto-detect dataset root!")
    print("   Listing contents of download directory:")
    !find {DOWNLOAD_DIR} -maxdepth 3 -type d | head -30
    print("\n   Please set Config.DATA_DIR manually in the next cell.")

✅ Kaggle API token configured!

⏳ Downloading dataset: ipythonx/mvtec-ad
   This is ~5 GB and may take 2-5 minutes...

Dataset URL: https://www.kaggle.com/datasets/ipythonx/mvtec-ad
License(s): copyright-authors
100% 4.91G/4.91G [05:49<00:00, 15.1MB/s]


✅ Download complete!

⏳ Extracting 1 zip file(s)...
   Extracting: mvtec-ad.zip
✅ Extraction complete!

✅ Dataset root auto-detected: /content/mvtec_ad
✅ Found 15 categories: ['bottle', 'cable', 'capsule', 'carpet', 'grid', 'hazelnut', 'leather', 'metal_nut', 'pill', 'screw', 'tile', 'toothbrush', 'transistor', 'wood', 'zipper']


---
## Cell 3: Import All Libraries

In [4]:
# ============================================================
# CELL 3: Import all required libraries
# ============================================================

import os
import json
import time
import numpy as np
import torch
import torch.nn.functional as F
from transformers import CLIPModel, CLIPProcessor
from PIL import Image
from scipy.ndimage import gaussian_filter
from sklearn.metrics import roc_auc_score, precision_recall_curve
from tqdm import tqdm

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device      : {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch version : 2.10.0+cu128
CUDA available  : True
GPU Device      : Tesla T4
GPU Memory      : 15.6 GB


---
## Cell 4: Configuration
Central configuration class adapted for Colab + GPU. The `DATA_DIR` is auto-set from the Kaggle download.

In [5]:
# ============================================================
# CELL 4: Configuration (adapted from src/config.py)
# ============================================================

class Config:
    # ---- PATHS ----
    # Auto-detected from Kaggle download in Cell 2.
    # If auto-detection failed, set this manually, e.g.:
    #   DATA_DIR = '/content/mvtec_ad'
    DATA_DIR = detected_root if detected_root else '/content/mvtec_ad'

    # Output directory (Colab runtime storage)
    OUTPUT_DIR = '/content/output'

    # ---- DEVICE ----
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

    # ---- ML HYPERPARAMETERS ----
    MODEL_NAME = 'openai/clip-vit-base-patch32'

    # GPU can handle much larger batch sizes than CPU
    BATCH_SIZE = 64

    # ---- WinCLIP SPECIFIC ----
    WINDOW_SIZE = (224, 224)   # Standard CLIP input size
    STRIDE = 56               # 75% overlap for dense heatmap

    # ---- NUMBER OF TEST IMAGES PER SUBFOLDER ----
    # GPU is fast — we use 30 images per subfolder for robust metrics.
    # Set to None to use ALL images (slower but most accurate).
    IMAGES_PER_SUBFOLDER = 30

    # ---- MVTEC AD CATEGORIES (15) ----
    OBJECTS = [
        'bottle', 'cable', 'capsule', 'carpet', 'grid', 'hazelnut',
        'leather', 'metal_nut', 'pill', 'screw', 'tile',
        'toothbrush', 'transistor', 'wood', 'zipper'
    ]

    # ---- ADVANCED PROMPT ENGINEERING ----
    NORMAL_STATES = [
        'normal', 'perfect', 'flawless', 'damage-free',
        'pristine', 'clean', 'good condition'
    ]
    DEFECT_STATES = [
        'damaged', 'broken', 'cracked', 'scratched', 'faulty',
        'defective', 'cut', 'hole', 'anomaly', 'blemish'
    ]
    TEMPLATES = [
        'a photo of a {}',
        'a close-up photo of a {}',
        'a cropped photo of a {}',
        'a good photo of a {}',
        'a bright photo of a {}',
        'a dark photo of a {}',
        'a bad photo of a {}',
        'a blurry photo of a {}'
    ]

# Create output directory
os.makedirs(Config.OUTPUT_DIR, exist_ok=True)

# Validation
print(f"\u2705 Device             : {Config.DEVICE}")
print(f"\u2705 Batch Size         : {Config.BATCH_SIZE}")
print(f"\u2705 Images/Subfolder   : {Config.IMAGES_PER_SUBFOLDER}")
print(f"\u2705 Dataset Path       : {Config.DATA_DIR}")
print(f"\u2705 Dataset exists     : {os.path.exists(Config.DATA_DIR)}")

if os.path.exists(Config.DATA_DIR):
    cats = sorted([
        d for d in os.listdir(Config.DATA_DIR)
        if os.path.isdir(os.path.join(Config.DATA_DIR, d))
    ])
    print(f"\u2705 Categories found   : {len(cats)} \u2192 {cats}")
else:
    print("\u274c Dataset NOT found! Check Cell 2 output.")

✅ Device             : cuda
✅ Batch Size         : 64
✅ Images/Subfolder   : 30
✅ Dataset Path       : /content/mvtec_ad
✅ Dataset exists     : True
✅ Categories found   : 15 → ['bottle', 'cable', 'capsule', 'carpet', 'grid', 'hazelnut', 'leather', 'metal_nut', 'pill', 'screw', 'tile', 'toothbrush', 'transistor', 'wood', 'zipper']


---
## Cell 5: WinCLIP Model
The core zero-shot anomaly detection model. Adapted from `src/models/winclip.py` with GPU optimizations.

In [6]:
# ============================================================
# CELL 5: WinCLIP Model (adapted from src/models/winclip.py)
# ============================================================
# Key GPU optimizations over the local CPU version:
#   - Model runs on CUDA
#   - Batch size increased from 16 to 64
#   - model.eval() for inference mode
# ============================================================

class WinCLIP:
    def __init__(self):
        self.device = Config.DEVICE
        print(f'Loading WinCLIP model on {self.device}...')
        self.model = CLIPModel.from_pretrained(Config.MODEL_NAME).to(self.device)
        self.processor = CLIPProcessor.from_pretrained(Config.MODEL_NAME)
        self.model.eval()  # Set to evaluation mode
        print('\u2705 Model loaded successfully!')

    def encode_text(self, category):
        """Generates text embeddings for normal and defect states using prompt ensembling."""
        # 1. Generate text prompts
        normal_prompts = [
            tpl.format(f'{state} {category}')
            for state in Config.NORMAL_STATES
            for tpl in Config.TEMPLATES
        ]
        defect_prompts = [
            tpl.format(f'{state} {category}')
            for state in Config.DEFECT_STATES
            for tpl in Config.TEMPLATES
        ]

        all_prompts = normal_prompts + defect_prompts

        # 2. Tokenize and Encode
        inputs = self.processor(
            text=all_prompts, return_tensors='pt', padding=True
        ).to(self.device)

        with torch.no_grad():
            outputs = self.model.get_text_features(**inputs)
            if hasattr(outputs, 'pooler_output'):
                text_features = outputs.pooler_output
            else:
                text_features = outputs

        # 3. Normalize
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)

        # 4. Average embeddings for Ensemble
        n_normal = len(normal_prompts)
        normal_features = text_features[:n_normal].mean(dim=0, keepdim=True)
        defect_features = text_features[n_normal:].mean(dim=0, keepdim=True)

        # Re-normalize after averaging
        normal_features = normal_features / normal_features.norm(dim=-1, keepdim=True)
        defect_features = defect_features / defect_features.norm(dim=-1, keepdim=True)

        # Stack: [2, Feature_Dim] -> Index 0: Normal, Index 1: Defect
        return torch.cat([normal_features, defect_features], dim=0)

    def extract_windows(self, image):
        """Slides a window across the image and returns patches."""
        W_h, W_w = Config.WINDOW_SIZE
        stride = Config.STRIDE

        w, h = image.size
        patches = []
        coordinates = []  # (x, y)

        for y in range(0, h - W_h + 1, stride):
            for x in range(0, w - W_w + 1, stride):
                patch = image.crop((x, y, x + W_w, y + W_h))
                patches.append(patch)
                coordinates.append((x, y))

        return patches, coordinates, (w, h)

    def predict(self, image, category='object'):
        """
        Runs WinCLIP inference.
        Returns:
            heatmap: 2D numpy array of anomaly scores.
            score: Max anomaly score (float).
        """
        # 1. Prepare Text Embeddings
        text_embedding = self.encode_text(category)  # shape [2, 512]

        # 2. Extract Patches (Local Features)
        patches, coords, img_dims = self.extract_windows(image)

        if not patches:
            # Image smaller than window size
            patches = [image.resize(Config.WINDOW_SIZE)]
            coords = [(0, 0)]

        # Process patches in batches (GPU can handle large batches)
        batch_size = Config.BATCH_SIZE
        patch_scores = []

        for i in range(0, len(patches), batch_size):
            batch_patches = patches[i:i + batch_size]
            inputs = self.processor(
                images=batch_patches, return_tensors='pt'
            ).to(self.device)

            with torch.no_grad():
                outputs = self.model.get_image_features(**inputs)
                if hasattr(outputs, 'pooler_output'):
                    image_features = outputs.pooler_output
                else:
                    image_features = outputs

            image_features = image_features / image_features.norm(dim=-1, keepdim=True)

            # Similarity: [Batch, 512] @ [2, 512].T -> [Batch, 2]
            # Index 1 is the 'Defect' score
            similarity = (100.0 * image_features @ text_embedding.T).softmax(dim=-1)
            batch_scores = similarity[:, 1].cpu().numpy()
            patch_scores.extend(batch_scores)

        # 3. Construct Heatmap
        w, h = img_dims
        heatmap = np.zeros((h, w))
        count_map = np.zeros((h, w))

        W_h, W_w = Config.WINDOW_SIZE

        for score_val, (x, y) in zip(patch_scores, coords):
            heatmap[y:y + W_h, x:x + W_w] += score_val
            count_map[y:y + W_h, x:x + W_w] += 1

        # Average overlap
        heatmap = np.divide(
            heatmap, count_map,
            out=np.zeros_like(heatmap), where=count_map != 0
        )

        # AU-ROC BOOST: Gaussian Smoothing
        # Smoothing raw patch scores removes isolated false-positive noise spikes
        heatmap = gaussian_filter(heatmap, sigma=4)

        # Calculate max score BEFORE normalization for image-level classification
        max_score = heatmap.max()

        # Normalize heatmap 0-1 for visualization
        if heatmap.max() > 0:
            heatmap = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min())

        return heatmap, max_score

print('\u2705 WinCLIP class defined.')

✅ WinCLIP class defined.


---
## Cell 6: Evaluation Function
Evaluates WinCLIP on a single MVTec AD category. Computes AU-ROC and optimal F1-Score. Adapted from `src/evaluate.py`.

In [7]:
# ============================================================
# CELL 6: Evaluation Function (adapted from src/evaluate.py)
# ============================================================
# Key change: accepts a pre-loaded model to avoid reloading
# the CLIP weights 15 times (once per category).
# ============================================================

def evaluate_category(category, model, limit=None):
    """
    Evaluate WinCLIP on a single MVTec AD category.

    Args:
        category (str): MVTec AD category name (e.g., 'bottle')
        model (WinCLIP): Pre-loaded WinCLIP model instance
        limit (int): Max images per subfolder. None = use all.

    Returns:
        tuple: (auroc, best_f1) or None if evaluation fails
    """
    print(f'\n{"=" * 60}')
    print(f'  Evaluating Category: {category}')
    print(f'{"=" * 60}')

    # Paths
    test_dir = os.path.join(Config.DATA_DIR, category, 'test')
    if not os.path.exists(test_dir):
        print(f'  \u26a0\ufe0f Test directory not found: {test_dir}')
        return None

    y_true = []
    y_scores = []

    # Iterate through subfolders
    # 'good' -> Label 0 (normal)
    # others  -> Label 1 (defect)
    subfolders = [
        d for d in os.listdir(test_dir)
        if os.path.isdir(os.path.join(test_dir, d))
    ]

    for subfolder in subfolders:
        is_defect = 1 if subfolder != 'good' else 0
        folder_path = os.path.join(test_dir, subfolder)

        images = [
            f for f in os.listdir(folder_path)
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))
        ]

        # Limit number of images per subfolder
        if limit:
            images = images[:limit]

        print(f'  Processing \'{subfolder}\' ({len(images)} images, label={is_defect})...')

        for img_name in tqdm(images, desc=f'    {subfolder}', leave=False):
            try:
                img_path = os.path.join(folder_path, img_name)
                image = Image.open(img_path).convert('RGB')

                # Predict
                _, score = model.predict(image, category)

                y_true.append(is_defect)
                y_scores.append(score)
            except Exception as e:
                print(f'    Error processing {img_name}: {e}')

    # Calculate Metrics
    if not y_true:
        print('  No data found.')
        return None

    y_true = np.array(y_true)
    y_scores = np.array(y_scores)

    # Check if we have both classes (normal + defect)
    if len(np.unique(y_true)) < 2:
        print('  Dataset contains only one class. Cannot calculate AU-ROC.')
        print(f'  Avg Score: {np.mean(y_scores):.4f}')
        return None

    # AU-ROC
    try:
        auroc = roc_auc_score(y_true, y_scores)
    except ValueError as e:
        print(f'  Error calculating AU-ROC: {e}')
        auroc = 0.0

    # Optimal F1-Score (finding best threshold)
    precision, recall, thresholds = precision_recall_curve(y_true, y_scores)
    f1_scores = 2 * recall * precision / (recall + precision + 1e-10)
    best_f1 = np.max(f1_scores)
    best_thresh = thresholds[np.argmax(f1_scores)]

    print(f'\n  \u2705 Results for {category}:')
    print(f'     AU-ROC     : {auroc:.4f}')
    print(f'     Best F1    : {best_f1:.4f} (threshold: {best_thresh:.4f})')
    print(f'     Samples    : {len(y_true)} (normal: {(y_true==0).sum()}, defect: {(y_true==1).sum()})')

    return auroc, best_f1

print('\u2705 evaluate_category() defined.')

✅ evaluate_category() defined.


---
## Cell 7: Model Comparison Generator
Runs evaluation across all categories and builds the comparative JSON. Adapted from `src/create_model_comparison.py`.

In [8]:
# ============================================================
# CELL 7: Model Comparison (from src/create_model_comparison.py)
# ============================================================

def create_model_comparison():
    print('=' * 70)
    print('  ZERO-SHOT DEFECT DETECTION \u2014 FULL MODEL BENCHMARKING')
    print('  Running WinCLIP on MVTec AD with Colab GPU Acceleration')
    print('=' * 70)

    dataset_dir = Config.DATA_DIR
    limit = Config.IMAGES_PER_SUBFOLDER

    if not os.path.exists(dataset_dir):
        print(f'\n\u274c Dataset directory not found: {dataset_dir}')
        print('Please check Cell 2 for download errors.')
        return None

    categories = [
        d for d in sorted(os.listdir(dataset_dir))
        if os.path.isdir(os.path.join(dataset_dir, d))
    ]
    print(f'\nFound {len(categories)} categories: {categories}')
    print(f'Images per subfolder limit: {limit}')

    # Load model ONCE (not per category)
    print('\n--- Loading WinCLIP model (one-time) ---')
    model = WinCLIP()

    aurocs = []
    f1s = []
    per_category_results = {}
    total_start = time.time()

    for i, category in enumerate(categories, 1):
        print(f'\n\u23f3 [{i}/{len(categories)}] Starting {category}...')
        cat_start = time.time()

        try:
            result = evaluate_category(category, model, limit=limit)
            if result:
                auc, f1 = result
                aurocs.append(auc)
                f1s.append(f1)
                per_category_results[category] = {
                    'AU-ROC': round(auc, 4),
                    'F1-Score': round(f1, 4)
                }
        except Exception as e:
            print(f'  [WARN] Failed to evaluate \'{category}\': {e}')

        cat_elapsed = time.time() - cat_start
        print(f'  \u23f1\ufe0f {category} completed in {cat_elapsed:.1f}s')

    total_elapsed = time.time() - total_start

    local_auroc = float(np.mean(aurocs)) if aurocs else 0.0
    local_f1 = float(np.mean(f1s)) if f1s else 0.0

    print(f'\n{"=" * 70}')
    print(f'  FINAL RESULTS \u2014 WinCLIP across {len(aurocs)}/{len(categories)} categories')
    print(f'  Average AU-ROC   : {local_auroc:.4f}')
    print(f'  Average F1-Score : {local_f1:.4f}')
    print(f'  Total Time       : {total_elapsed:.1f}s ({total_elapsed/60:.1f}min)')
    print(f'{"=" * 70}\n')

    # Per-category breakdown
    print('\n  Per-Category Results:')
    print(f'  {"Category":<15} {"AU-ROC":>10} {"F1-Score":>10}')
    print(f'  {"-" * 35}')
    for cat, res in per_category_results.items():
        print(f'  {cat:<15} {res["AU-ROC"]:>10.4f} {res["F1-Score"]:>10.4f}')
    print(f'  {"-" * 35}')
    print(f'  {"AVERAGE":<15} {local_auroc:>10.4f} {local_f1:>10.4f}\n')

    # -------- Build Comparative JSON (same as original) --------
    models_data = [
        {
            'Model_Name': 'WinCLIP (this project)',
            'Core_Architecture': 'Window-based CLIP with compositional prompt ensemble',
            'Key_Strengths': 'Sliding window analysis excels at finding fine structural anomalies and missing components (screws, transistors).',
            'Limitations': 'Lower global classification AUROC compared to newer methods; slower on CPU.',
            'Best_Use_Case': 'Structural defects, logical anomalies, and small missing components on rigid objects.',
            'Zero_Shot_Performance_Notes': (
                f'Evaluated on MVTec AD ({len(aurocs)} categories, '
                f'{limit} samples/sub-folder, GPU): '
                f'avg AU-ROC = {local_auroc:.4f}, avg F1-Score = {local_f1:.4f}.'
            ),
            'Per_Category_Results': per_category_results
        },
        {
            'Model_Name': 'AnomalyCLIP',
            'Core_Architecture': 'Object-agnostic text prompt learning with CLIP backbone',
            'Key_Strengths': 'Learns generalizable, object-agnostic prompts \u2014 excels on texture-heavy categories (carpet, tile).',
            'Limitations': 'Less specialized for fine structural / logical anomalies.',
            'Best_Use_Case': 'Texture-based defect detection across a wide range of object types.',
            'Zero_Shot_Performance_Notes': 'Benchmark (MVTec AD): ~0.916 Classification AUROC, ~0.907 Segmentation AUROC. Significantly outperforms WinCLIP on global metrics.'
        },
        {
            'Model_Name': 'AdaptCLIP',
            'Core_Architecture': 'Hybrid static + dynamic learnable prompts with alternating visual/textual optimization',
            'Key_Strengths': 'Adapts to each test image dynamically without any training data; jointly optimizes both modalities.',
            'Limitations': 'Computationally heavier at inference due to dynamic adaptation steps.',
            'Best_Use_Case': 'High-variability appearance settings where training is not available.',
            'Zero_Shot_Performance_Notes': 'Benchmark: ~89.6% pixel-AUROC in zero-shot settings on MVTec AD.'
        },
        {
            'Model_Name': 'AnomalyGPT',
            'Core_Architecture': 'LLM integrated with a Vision-Language Model (VLM)',
            'Key_Strengths': 'No manual threshold tuning required; supports multi-turn textual reasoning and natural language defect explanations.',
            'Limitations': 'High computational cost from LLM inference; latency unsuitable for real-time industrial lines.',
            'Best_Use_Case': 'Interactive, explainable inspection pipelines where human-readable reasoning is required.',
            'Zero_Shot_Performance_Notes': 'Benchmark: Leverages large-scale LLM world knowledge for superior reasoning without manual calibration.'
        },
        {
            'Model_Name': 'FiLo',
            'Core_Architecture': 'LLM + Grounding DINO for fine-grained visual grounding',
            'Key_Strengths': 'Targets micro-defects by grounding detailed textual descriptions to precise image regions.',
            'Limitations': 'Requires multiple heavy models (LLM + Grounding DINO), increasing deployment complexity.',
            'Best_Use_Case': 'Micro-defect detection where exact text-to-region grounding is critical.',
            'Zero_Shot_Performance_Notes': 'Benchmark: State-of-the-art on fine-grained anomaly detection benchmarks.'
        }
    ]

    # Save JSON
    output_path = os.path.join(Config.OUTPUT_DIR, 'model_comparison.json')
    with open(output_path, 'w') as f:
        json.dump(models_data, f, indent=4)

    print(f'\u2705 Saved comparison JSON to: {output_path}')

    return models_data

print('\u2705 create_model_comparison() defined.')

✅ create_model_comparison() defined.


---
## Cell 8: RUN BENCHMARKING 🚀
Execute the full evaluation pipeline. This will:
1. Load the WinCLIP model once on GPU
2. Iterate through all 15 MVTec AD categories
3. Compute AU-ROC and F1-Score for each
4. Save `model_comparison.json`

> **Estimated time on T4 GPU:** ~15–25 minutes for 30 images/subfolder across 15 categories

In [9]:
# ============================================================
# CELL 8: Run the full benchmarking pipeline
# ============================================================

results = create_model_comparison()

  ZERO-SHOT DEFECT DETECTION — FULL MODEL BENCHMARKING
  Running WinCLIP on MVTec AD with Colab GPU Acceleration

Found 15 categories: ['bottle', 'cable', 'capsule', 'carpet', 'grid', 'hazelnut', 'leather', 'metal_nut', 'pill', 'screw', 'tile', 'toothbrush', 'transistor', 'wood', 'zipper']
Images per subfolder limit: 30

--- Loading WinCLIP model (one-time) ---
Loading WinCLIP model on cuda...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

...

  Processing 'rough' (17 images, label=1)...


  Processing 'fabric_border' (17 images, label=1)...


  Processing 'good' (30 images, label=0)...


  Processing 'broken_teeth' (19 images, label=1)...



  ✅ Results for zipper:
     AU-ROC     : 0.6482
     Best F1    : 0.8914 (threshold: 0.6115)
     Samples    : 149 (normal: 30, defect: 119)
  ⏱️ zipper completed in 153.0s

  FINAL RESULTS — WinCLIP across 15/15 categories
  Average AU-ROC   : 0.7155
  Average F1-Score : 0.8903
  Total Time       : 1514.7s (25.2min)


  Per-Category Results:
  Category            AU-ROC   F1-Score
  -----------------------------------
  bottle              0.9349     0.9333
  cable               0.5569     0.8638
  capsule             0.6968     0.9153
  carpet              0.6778     0.8641
  grid                0.7886     0.8636
  hazelnut            0.7567     0.8284
  leather             0.9500     0.9399
  metal_nut           0.3377     0.8942
  pill                0.3342     0.9156
  screw               0.8756     0.9243
  tile                0.9433     0.9412
  toothbrush          0.5722     0.8406
  transistor          0.6842     0.7723
  wood                0.9754     0.9672
  zipper       

---
## Cell 9: Display Results

In [10]:
# ============================================================
# CELL 9: Display the generated JSON
# ============================================================

if results:
    print('\n' + '=' * 70)
    print('  GENERATED model_comparison.json')
    print('=' * 70 + '\n')
    print(json.dumps(results, indent=4))
else:
    print('No results generated. Check dataset path.')


  GENERATED model_comparison.json

[
    {
        "Model_Name": "WinCLIP (this project)",
        "Core_Architecture": "Window-based CLIP with compositional prompt ensemble",
        "Key_Strengths": "Sliding window analysis excels at finding fine structural anomalies and missing components (screws, transistors).",
        "Limitations": "Lower global classification AUROC compared to newer methods; slower on CPU.",
        "Best_Use_Case": "Structural defects, logical anomalies, and small missing components on rigid objects.",
        "Zero_Shot_Performance_Notes": "Evaluated on MVTec AD (15 categories, 30 samples/sub-folder, GPU): avg AU-ROC = 0.7155, avg F1-Score = 0.8903.",
        "Per_Category_Results": {
            "bottle": {
                "AU-ROC": 0.9349,
                "F1-Score": 0.9333
            },
            "cable": {
                "AU-ROC": 0.5569,
                "F1-Score": 0.8638
            },
            "capsule": {
                "AU-ROC": 0.6968,
    

---
## Cell 10: Download the Output JSON
Downloads the generated `model_comparison.json` to your local machine.

In [11]:
# ============================================================
# CELL 10: Download the output JSON file
# ============================================================

from google.colab import files

output_path = os.path.join(Config.OUTPUT_DIR, 'model_comparison.json')

if os.path.exists(output_path):
    files.download(output_path)
    print(f'\u2705 Downloading {output_path}...')
    print('\nPlace the downloaded file at:')
    print('  data/processed/model_comparison.json')
    print('in your local project to update the benchmarks.')
else:
    print('\u274c Output file not found. Run Cell 8 first.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Downloading /content/output/model_comparison.json...

Place the downloaded file at:
  data/processed/model_comparison.json
in your local project to update the benchmarks.
